In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn scikit-learn tensorflow keras ta -q

In [ ]:
# Import semua library
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data
import yfinance as yf
from datetime import datetime, timedelta

# Technical Indicators
import ta
from ta.momentum import RSIIndicator
from ta.trend import MACD, SMAIndicator

# Machine Learning
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Set random seed untuk reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Style plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"TensorFlow versi: {tf.__version__}")
print(f"Semua library berhasil diimport!")

In [ ]:
# ============================================================
# DOWNLOAD DATA BITCOIN DARI YAHOO FINANCE
# ============================================================

print("Mengunduh data Bitcoin dari Yahoo Finance...")

START_DATE = '2015-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')
TICKER     = 'BTC-USD'

df_raw = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=True)
df_raw = df_raw.reset_index()

# Flatten MultiIndex jika ada
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = [col[0] if col[1] == '' else col[0] for col in df_raw.columns]

print(f"\nData berhasil diunduh!")
print(f"   Periode  : {df_raw['Date'].min().date()} s/d {df_raw['Date'].max().date()}")
print(f"   Jumlah   : {len(df_raw):,} baris data")
print(f"   Kolom    : {list(df_raw.columns)}")

In [ ]:
# ============================================================
# TAMPILKAN INFO DATASET
# ============================================================

print("=" * 60)
print("INFORMASI DATASET")
print("=" * 60)
df_raw.info()

print("\n" + "=" * 60)
print("5 DATA PERTAMA")
print("=" * 60)
display(df_raw.head())

print("\n" + "=" * 60)
print("  STATISTIK DESKRIPTIF")
print("=" * 60)
display(df_raw.describe().round(2))

In [ ]:
# ============================================================
# CEK MISSING VALUES
# ============================================================

print("=" * 40)
print("CEK MISSING VALUES")
print("=" * 40)
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

if missing.sum() == 0:
    print("\nTidak ada missing values!")
else:
    print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# ============================================================
# VISUALISASI DATA AWAL
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 14))
fig.suptitle('Analisis Awal Data Bitcoin (BTC-USD)', fontsize=18, fontweight='bold', y=0.98)

# 1. Harga Penutupan
axes[0].plot(df_raw['Date'], df_raw['Close'], color='#F7931A', linewidth=1.5, label='Closing Price')
axes[0].fill_between(df_raw['Date'], df_raw['Close'], alpha=0.1, color='#F7931A')
axes[0].set_title('Harga Penutupan Bitcoin (USD)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Harga (USD)', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# 2. Volume Trading
axes[1].bar(df_raw['Date'], df_raw['Volume'], color='#4A90D9', alpha=0.7, label='Volume')
axes[1].set_title('Volume Trading Bitcoin', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Volume', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e9:.1f}B'))

# 3. Return Harian
daily_return = df_raw['Close'].pct_change() * 100
axes[2].plot(df_raw['Date'], daily_return, color='#2ECC71', linewidth=0.8, label='Daily Return (%)')
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[2].fill_between(df_raw['Date'], daily_return, 0,
                     where=(daily_return >= 0), color='#2ECC71', alpha=0.3, label='Positif')
axes[2].fill_between(df_raw['Date'], daily_return, 0,
                     where=(daily_return < 0), color='#E74C3C', alpha=0.3, label='Negatif')
axes[2].set_title('Return Harian (%)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Return (%)', fontsize=11)
axes[2].set_xlabel('Tanggal', fontsize=11)
axes[2].legend(fontsize=10)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gru_01_analisis_awal.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualisasi analisis awal selesai!")

In [ ]:
# ============================================================
# BUAT SALINAN DATA & BERSIHKAN
# ============================================================

df = df_raw.copy()
df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
df = df.sort_values('Date').reset_index(drop=True)
df = df.dropna()

print(f"Data dibersihkan: {len(df):,} baris")

In [ ]:
# ============================================================
# HITUNG INDIKATOR TEKNIKAL
# ============================================================

print("Menghitung indikator teknikal...")

# ------ Moving Average ------
df['MA7']  = SMAIndicator(close=df['Close'], window=7).sma_indicator()
df['MA21'] = SMAIndicator(close=df['Close'], window=21).sma_indicator()
df['MA50'] = SMAIndicator(close=df['Close'], window=50).sma_indicator()

# ------ RSI (14 periode default) ------
rsi = RSIIndicator(close=df['Close'], window=14)
df['RSI'] = rsi.rsi()

# ------ MACD ------
macd = MACD(close=df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD']        = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Hist']   = macd.macd_diff()

# ------ Fitur Tambahan dari Tanggal ------
df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Month']       = df['Date'].dt.month
df['Year']        = df['Date'].dt.year

# Hapus baris dengan NaN (akibat perhitungan window indikator)
df = df.dropna().reset_index(drop=True)

print(f"Indikator teknikal berhasil dihitung!")
print(f"Jumlah baris setelah kalkulasi: {len(df):,}")
print(f"Kolom akhir: {list(df.columns)}")

In [ ]:
# ============================================================
# VISUALISASI INDIKATOR TEKNIKAL
# ============================================================

# Ambil 2 tahun terakhir untuk visualisasi lebih jelas
df_viz = df.tail(730).copy()

fig, axes = plt.subplots(4, 1, figsize=(16, 18))
fig.suptitle('Indikator Teknikal Bitcoin', fontsize=18, fontweight='bold', y=0.99)

# 1. Harga + Moving Average
axes[0].plot(df_viz['Date'], df_viz['Close'], label='Close Price', color='#F7931A', linewidth=1.5)
axes[0].plot(df_viz['Date'], df_viz['MA7'],  label='MA7',  color='#3498DB', linewidth=1.2, linestyle='--')
axes[0].plot(df_viz['Date'], df_viz['MA21'], label='MA21', color='#9B59B6', linewidth=1.2, linestyle='--')
axes[0].plot(df_viz['Date'], df_viz['MA50'], label='MA50', color='#E74C3C', linewidth=1.2, linestyle='--')
axes[0].set_title('Harga Penutupan & Moving Average', fontweight='bold')
axes[0].set_ylabel('Harga (USD)')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# 2. Volume
colors = ['#2ECC71' if df_viz['Close'].iloc[i] >= df_viz['Close'].iloc[i-1] else '#E74C3C'
          for i in range(len(df_viz))]
axes[1].bar(df_viz['Date'], df_viz['Volume'], color=colors, alpha=0.7, label='Volume')
axes[1].set_title('Volume Trading', fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e9:.1f}B'))

# 3. RSI
axes[2].plot(df_viz['Date'], df_viz['RSI'], color='#8E44AD', linewidth=1.5, label='RSI')
axes[2].axhline(y=70, color='#E74C3C', linestyle='--', linewidth=1.2, label='Overbought (70)')
axes[2].axhline(y=30, color='#2ECC71', linestyle='--', linewidth=1.2, label='Oversold (30)')
axes[2].fill_between(df_viz['Date'], df_viz['RSI'], 70,
                     where=(df_viz['RSI'] >= 70), color='#E74C3C', alpha=0.2)
axes[2].fill_between(df_viz['Date'], df_viz['RSI'], 30,
                     where=(df_viz['RSI'] <= 30), color='#2ECC71', alpha=0.2)
axes[2].set_title('RSI (14)', fontweight='bold')
axes[2].set_ylabel('RSI')
axes[2].set_ylim(0, 100)
axes[2].legend(fontsize=9)

# 4. MACD
axes[3].plot(df_viz['Date'], df_viz['MACD'],        color='#3498DB', linewidth=1.5, label='MACD')
axes[3].plot(df_viz['Date'], df_viz['MACD_Signal'], color='#E74C3C', linewidth=1.2, linestyle='--', label='Signal')
hist_colors = ['#2ECC71' if v >= 0 else '#E74C3C' for v in df_viz['MACD_Hist']]
axes[3].bar(df_viz['Date'], df_viz['MACD_Hist'], color=hist_colors, alpha=0.6, label='Histogram')
axes[3].axhline(y=0, color='black', linewidth=0.8)
axes[3].set_title('MACD (12,26,9)', fontweight='bold')
axes[3].set_ylabel('MACD')
axes[3].set_xlabel('Tanggal')
axes[3].legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('gru_02_indikator_teknikal.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualisasi indikator teknikal selesai!")

In [ ]:
# ============================================================
# KORELASI ANTAR FITUR
# ============================================================

feature_cols = ['Close', 'Volume', 'MA7', 'MA21', 'MA50', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist']
corr_matrix = df[feature_cols].corr()

plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Matriks Korelasi Fitur', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('gru_03_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Matriks korelasi ditampilkan!")

In [ ]:
# ============================================================
# DEFINISI FITUR & TARGET
# ============================================================

FEATURES = ['Close', 'Volume', 'MA7', 'MA21', 'MA50', 'RSI',
            'MACD', 'MACD_Signal', 'MACD_Hist', 'Day_of_Week', 'Month']

TARGET = 'Close'

data = df[FEATURES].values

print(f"Fitur yang digunakan ({len(FEATURES)} fitur):")
for i, f in enumerate(FEATURES, 1):
    print(f"{i:2d}. {f}")
print(f"\nTarget: {TARGET}")
print(f"Shape data: {data.shape}")

In [ ]:
# ============================================================
# NORMALISASI DATA (FIT HANYA PADA DATA TRAINING)
# ============================================================
# Hitung batas split DULU sebelum normalisasi
LOOKBACK = 30  # Akan digunakan juga di cell berikutnya
n_total_raw = len(data)
TRAIN_RATIO = 0.70
# Offset karena lookback akan mengurangi jumlah sampel
n_train_raw = int((n_total_raw - LOOKBACK) * TRAIN_RATIO) + LOOKBACK
# Fit scaler HANYA pada data training (mencegah data leakage!)
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_X.fit(data[:n_train_raw])   # ← FIT hanya pada porsi training
scaled_data = scaler_X.transform(data)  # Transform semua data
scaler_y = MinMaxScaler(feature_range=(0, 1))
scaler_y.fit(df[[TARGET]].values[:n_train_raw])  # ← FIT hanya pada porsi training
scaled_close = scaler_y.transform(df[[TARGET]].values)
print(f"Normalisasi selesai (MinMaxScaler 0-1)")
print(f"PENTING: Scaler di-fit HANYA pada {n_train_raw} data training pertama")
print(f"Shape scaled: {scaled_data.shape}")
print(f"Min (training): {scaled_data[:n_train_raw].min():.4f}, Max (training): {scaled_data[:n_train_raw].max():.4f}")

In [ ]:
# ============================================================
# BUAT SEKUENS TIME SERIES UNTUK LSTM
# ============================================================

def create_sequences(data_X, data_y, lookback):
    """Buat sequence X dan y untuk LSTM."""
    X, y = [], []
    for i in range(lookback, len(data_X)):
        X.append(data_X[i - lookback:i])  # lookback hari sebelumnya
        y.append(data_y[i, 0])            # harga penutupan hari ini
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, scaled_close, LOOKBACK)

print(f"Sekuens berhasil dibuat!")
print(f"Lookback window : {LOOKBACK} hari")
print(f"Shape X         : {X.shape}  (samples, timesteps, features)")
print(f"Shape y         : {y.shape}  (samples,)")

In [ ]:
# ============================================================
# SPLIT DATA: TRAIN / VALIDATION / TEST
# ============================================================

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO  = 0.15

n_total = len(X)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
n_test  = n_total - n_train - n_val

X_train, y_train = X[:n_train],           y[:n_train]
X_val,   y_val   = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test,  y_test  = X[n_train+n_val:],     y[n_train+n_val:]

# Simpan index tanggal untuk visualisasi
dates = df['Date'].values[LOOKBACK:]
dates_train = dates[:n_train]
dates_val   = dates[n_train:n_train+n_val]
dates_test  = dates[n_train+n_val:]

print("Data berhasil di-split:")
print(f"Train      : {n_train:,} ({TRAIN_RATIO*100:.0f}%) | {str(dates_train[0])[:10]} s/d {str(dates_train[-1])[:10]}")
print(f"Validation : {n_val:,} ({VAL_RATIO*100:.0f}%) | {str(dates_val[0])[:10]} s/d {str(dates_val[-1])[:10]}")
print(f"Test       : {n_test:,} ({(1-TRAIN_RATIO-VAL_RATIO)*100:.0f}%) | {str(dates_test[0])[:10]} s/d {str(dates_test[-1])[:10]}")

In [ ]:
# ============================================================
# BANGUN ARSITEKTUR MODEL GRU
# ============================================================

def build_gru_model(input_shape, gru_units=[64, 32], dropout_rate=0.2, learning_rate=0.001):
    """
    Membangun model GRU multi-layer.

    Args:
        input_shape   : (timesteps, features)
        gru_units     : list jumlah unit per layer GRU
        dropout_rate  : dropout untuk regularisasi
        learning_rate : learning rate optimizer

    Returns:
        model         : Keras Sequential model
    """
    model = Sequential(name='GRU_Bitcoin_Predictor')

    # Layer GRU 1 - return_sequences=True karena ada layer berikutnya
    model.add(GRU(
        units=gru_units[0],
        return_sequences=True,
        input_shape=input_shape,
        name='GRU_1'
    ))
    model.add(Dropout(dropout_rate, name='Dropout_1'))

    # Layer GRU 2
    model.add(GRU(
        units=gru_units[1],
        return_sequences=True,
        name='GRU_2'
    ))
    model.add(Dropout(dropout_rate, name='Dropout_2'))

    # # Layer GRU 3 - return_sequences=False (terakhir)
    # model.add(GRU(
    #     units=gru_units[2],
    #     return_sequences=False,
    #     name='GRU_3'
    # ))
    # model.add(Dropout(dropout_rate / 2, name='Dropout_3'))

    # Dense layers
    model.add(Dense(units=16, activation='relu', name='Dense_1'))
    model.add(Dense(units=1, name='Output'))  # Prediksi 1 nilai

    # Compile
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model


# Build model
INPUT_SHAPE = (X_train.shape[1], X_train.shape[2])  # (60, 11)
model = build_gru_model(
    input_shape=INPUT_SHAPE,
    gru_units=[128, 64, 32],
    dropout_rate=0.2,
    learning_rate=0.001
)

model.summary()

In [ ]:
# ============================================================
# SETTING CALLBACKS
# ============================================================

callbacks = [
    # Hentikan training jika tidak ada peningkatan setelah 15 epoch
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        min_delta=0.0001,
        restore_best_weights=True,
        verbose=1
    ),
    # Simpan model terbaik
    ModelCheckpoint(
        filepath='best_gru_bitcoin.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    ),
    # Kurangi learning rate jika tidak ada peningkatan setelah 7 epoch
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=1
    )
]

print("Callbacks telah disiapkan:")
print("- EarlyStopping    : patience=20")
print("- ModelCheckpoint  : simpan model terbaik")
print("- ReduceLROnPlateau: patience=10, factor=0.5")

In [ ]:
# ============================================================
# LATIH MODEL
# ============================================================

EPOCHS     = 200
BATCH_SIZE = 32

print(f"Mulai melatih model GRU...")
print(f"Epochs     : {EPOCHS}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Input shape: {INPUT_SHAPE}")
print("=" * 60)

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
    shuffle=False  # Jangan shuffle untuk time series!
)

print("\nraining selesai!")
print(f"Total epoch dijalankan : {len(history.history['loss'])}")
print(f"Best val_loss          : {min(history.history['val_loss']):.6f}")

In [ ]:
# ============================================================
# VISUALISASI TRAINING HISTORY
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Training History Model GRU', fontsize=16, fontweight='bold')

epochs_ran = range(1, len(history.history['loss']) + 1)

# Loss
axes[0].plot(epochs_ran, history.history['loss'],     'b-', linewidth=2, label='Train Loss')
axes[0].plot(epochs_ran, history.history['val_loss'], 'r-', linewidth=2, label='Val Loss')
axes[0].set_title('Loss (MSE) per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(epochs_ran, history.history['mae'],     'b-', linewidth=2, label='Train MAE')
axes[1].plot(epochs_ran, history.history['val_mae'], 'r-', linewidth=2, label='Val MAE')
axes[1].set_title('MAE per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gru_04_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training history divisualisasikan!")

In [ ]:
# ============================================================
# FUNGSI METRIK EVALUASI
# ============================================================

def calculate_metrics(y_true, y_pred, label=''):
    """
    Hitung RMSE, MAE, MAPE, dan Akurasi.

    Returns:
        dict berisi semua metrik
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # MAPE: hindari pembagian nol
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    # Akurasi = 100% - MAPE
    accuracy = max(0, 100 - mape)

    if label:
        print(f"\n{'='*50}")
        print(f"  EVALUASI MODEL - {label}")
        print(f"{'='*50}")
        print(f"  RMSE     : ${rmse:>12,.2f}")
        print(f"  MAE      : ${mae:>12,.2f}")
        print(f"  MAPE     : {mape:>11.4f} %")
        print(f"  Akurasi  : {accuracy:>11.4f} %")
        print(f"{'='*50}")

    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'Accuracy': accuracy}

print("Fungsi metrik evaluasi siap!")

In [ ]:
# ============================================================
# PREDIKSI & INVERSE TRANSFORM
# ============================================================

print("Melakukan prediksi...")

# Prediksi (masih dalam skala 0-1)
y_pred_train_scaled = model.predict(X_train, verbose=0)
y_pred_val_scaled   = model.predict(X_val,   verbose=0)
y_pred_test_scaled  = model.predict(X_test,  verbose=0)

# Inverse transform ke harga asli (USD)
y_pred_train = scaler_y.inverse_transform(y_pred_train_scaled.reshape(-1, 1)).flatten()
y_pred_val   = scaler_y.inverse_transform(y_pred_val_scaled.reshape(-1, 1)).flatten()
y_pred_test  = scaler_y.inverse_transform(y_pred_test_scaled.reshape(-1, 1)).flatten()

y_true_train = scaler_y.inverse_transform(y_train.reshape(-1, 1)).flatten()
y_true_val   = scaler_y.inverse_transform(y_val.reshape(-1, 1)).flatten()
y_true_test  = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

print("Prediksi selesai!")

In [ ]:
# ============================================================
# HITUNG SEMUA METRIK
# ============================================================

metrics_train = calculate_metrics(y_true_train, y_pred_train, label='TRAIN SET')
metrics_val   = calculate_metrics(y_true_val,   y_pred_val,   label='VALIDATION SET')
metrics_test  = calculate_metrics(y_true_test,  y_pred_test,  label='TEST SET')

# Ringkasan tabel
metrics_df = pd.DataFrame({
    'Dataset'   : ['Train', 'Validation', 'Test'],
    'RMSE (USD)': [f"${metrics_train['RMSE']:,.2f}",
                   f"${metrics_val['RMSE']:,.2f}",
                   f"${metrics_test['RMSE']:,.2f}"],
    'MAE (USD)' : [f"${metrics_train['MAE']:,.2f}",
                   f"${metrics_val['MAE']:,.2f}",
                   f"${metrics_test['MAE']:,.2f}"],
    'MAPE (%)'  : [f"{metrics_train['MAPE']:.4f}%",
                   f"{metrics_val['MAPE']:.4f}%",
                   f"{metrics_test['MAPE']:.4f}%"],
    'Akurasi (%)': [f"{metrics_train['Accuracy']:.4f}%",
                    f"{metrics_val['Accuracy']:.4f}%",
                    f"{metrics_test['Accuracy']:.4f}%"]
})

print("\n" + "="*60)
print("  RINGKASAN EVALUASI MODEL GRU")
print("="*60)
display(metrics_df.set_index('Dataset'))

In [ ]:
# ============================================================
# VISUALISASI PREDIKSI vs AKTUAL
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(18, 18))
fig.suptitle('Hasil Prediksi GRU vs Harga Aktual Bitcoin', fontsize=18, fontweight='bold', y=0.99)

# Konversi tanggal
dates_train_pd = pd.to_datetime(dates_train)
dates_val_pd   = pd.to_datetime(dates_val)
dates_test_pd  = pd.to_datetime(dates_test)

datasets = [
    ('Train',      dates_train_pd, y_true_train, y_pred_train, metrics_train, '#3498DB', '#E74C3C'),
    ('Validation', dates_val_pd,   y_true_val,   y_pred_val,   metrics_val,   '#2ECC71', '#E67E22'),
    ('Test',       dates_test_pd,  y_true_test,  y_pred_test,  metrics_test,  '#9B59B6', '#1ABC9C'),
]

for idx, (name, dates_pd, y_true, y_pred, metrics, c_true, c_pred) in enumerate(datasets):
    ax = axes[idx]
    ax.plot(dates_pd, y_true, color=c_true, linewidth=1.5, label='Aktual', alpha=0.9)
    ax.plot(dates_pd, y_pred, color=c_pred, linewidth=1.5, label='Prediksi', alpha=0.9, linestyle='--')
    ax.fill_between(dates_pd, y_true, y_pred, alpha=0.1, color='gray', label='Error')
    ax.set_title(
        f'{name} | RMSE: ${metrics["RMSE"]:,.0f} | MAE: ${metrics["MAE"]:,.0f} | '
        f'MAPE: {metrics["MAPE"]:.2f}% | Akurasi: {metrics["Accuracy"]:.2f}%',
        fontsize=11, fontweight='bold'
    )
    ax.set_ylabel('Harga Bitcoin (USD)')
    ax.legend(loc='upper left', fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

axes[-1].set_xlabel('Tanggal', fontsize=12)
plt.tight_layout()
plt.savefig('gru_05_prediksi_vs_aktual.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualisasi prediksi selesai!")

In [ ]:
# ============================================================
# VISUALISASI DETAIL TEST SET + SCATTER PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Analisis Detail Hasil Test Set', fontsize=16, fontweight='bold')

# 1. Prediksi vs Aktual Test
axes[0].plot(dates_test_pd, y_true_test, 'b-', linewidth=1.8, label='Aktual', alpha=0.9)
axes[0].plot(dates_test_pd, y_pred_test, 'r--', linewidth=1.8, label='Prediksi', alpha=0.9)
axes[0].fill_between(dates_test_pd, y_true_test, y_pred_test, alpha=0.15, color='orange')
axes[0].set_title('Prediksi vs Aktual (Test Set)', fontweight='bold')
axes[0].set_ylabel('Harga Bitcoin (USD)')
axes[0].set_xlabel('Tanggal')
axes[0].legend(fontsize=11)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(True, alpha=0.3)

# 2. Scatter Plot
axes[1].scatter(y_true_test, y_pred_test, alpha=0.5, color='#3498DB', edgecolors='white',
                linewidths=0.5, s=40, label='Data Point')
min_val = min(y_true_test.min(), y_pred_test.min())
max_val = max(y_true_test.max(), y_pred_test.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Garis Ideal')
axes[1].set_title('Scatter: Prediksi vs Aktual', fontweight='bold')
axes[1].set_xlabel('Harga Aktual (USD)')
axes[1].set_ylabel('Harga Prediksi (USD)')
axes[1].legend(fontsize=11)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
axes[1].grid(True, alpha=0.3)

# Korelasi
corr = np.corrcoef(y_true_test, y_pred_test)[0, 1]
axes[1].text(0.05, 0.95, f'r = {corr:.4f}', transform=axes[1].transAxes,
             fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.savefig('gru_06_detail_test.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Korelasi Pearson (Test): r = {corr:.4f}")

In [ ]:
# ============================================================
# VISUALISASI DISTRIBUSI ERROR
# ============================================================

errors = y_true_test - y_pred_test
errors_pct = ((y_true_test - y_pred_test) / y_true_test) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Distribusi Error Prediksi (Test Set)', fontsize=15, fontweight='bold')

axes[0].hist(errors, bins=50, color='#3498DB', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].axvline(x=errors.mean(), color='orange', linestyle='--', linewidth=2,
                label=f'Mean Error: ${errors.mean():,.0f}')
axes[0].set_title('Distribusi Residual (USD)', fontweight='bold')
axes[0].set_xlabel('Error (USD)')
axes[0].set_ylabel('Frekuensi')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].hist(errors_pct, bins=50, color='#E74C3C', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='blue', linestyle='--', linewidth=2, label='Zero Error')
axes[1].axvline(x=errors_pct.mean(), color='orange', linestyle='--', linewidth=2,
                label=f'Mean Error: {errors_pct.mean():.2f}%')
axes[1].set_title('Distribusi Error Persentase (%)', fontweight='bold')
axes[1].set_xlabel('Error (%)')
axes[1].set_ylabel('Frekuensi')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gru_07_distribusi_error.png', dpi=150, bbox_inches='tight')
plt.show()
print("Distribusi error divisualisasikan!")

In [ ]:
# ============================================================
# PREDIKSI N HARI KE DEPAN
# ============================================================

def predict_future(model, last_sequence, scaler_X, scaler_y, n_days=30, n_features=11):
    """
    Prediksi harga N hari ke depan secara iteratif.

    Args:
        last_sequence : sekuens terakhir dari data (shape: lookback x n_features)
        n_days        : berapa hari ke depan yang ingin diprediksi
    """
    current_seq = last_sequence.copy()
    future_preds = []

    for _ in range(n_days):
        # Prediksi satu langkah
        pred_scaled = model.predict(current_seq.reshape(1, *current_seq.shape), verbose=0)
        pred_price  = scaler_y.inverse_transform(pred_scaled)[0, 0]
        future_preds.append(pred_price)

        # Update sekuens: geser 1 langkah, isi dengan prediksi baru
        new_row       = current_seq[-1].copy()
        # Update fitur Close (index 0) dengan prediksi terbaru (dalam skala)
        new_row[0]    = pred_scaled[0, 0]
        current_seq   = np.vstack([current_seq[1:], new_row])

    return np.array(future_preds)


# Ambil sekuens terakhir dari seluruh data
last_seq = scaled_data[-LOOKBACK:]
FUTURE_DAYS = 30

print(f"🔮 Memprediksi {FUTURE_DAYS} hari ke depan...")
future_prices = predict_future(model, last_seq, scaler_X, scaler_y,
                                n_days=FUTURE_DAYS, n_features=len(FEATURES))

# Buat tanggal masa depan
last_date    = df['Date'].iloc[-1]
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=FUTURE_DAYS)

future_df = pd.DataFrame({'Date': future_dates, 'Predicted_Price': future_prices})

print(f"  PREDIKSI {FUTURE_DAYS} HARI KE DEPAN")
print(future_df.to_string(index=False))

In [ ]:
# ============================================================
# VISUALISASI PREDIKSI MASA DEPAN
# ============================================================

# Ambil 90 hari terakhir data historis
hist_days = 90
df_recent = df.tail(hist_days)

plt.figure(figsize=(16, 8))

# Data historis
plt.plot(df_recent['Date'], df_recent['Close'],
         color='#3498DB', linewidth=2, label='Harga Historis', alpha=0.9)

# Prediksi masa depan
plt.plot(future_dates, future_prices,
         color='#E74C3C', linewidth=2, linestyle='--', label='Prediksi Masa Depan', alpha=0.9)

# Shaded area prediksi (±5% uncertainty)
plt.fill_between(future_dates,
                 future_prices * 0.95,
                 future_prices * 1.05,
                 alpha=0.2, color='#E74C3C', label='Uncertainty ±5%')

# Garis pemisah historis vs prediksi
plt.axvline(x=last_date, color='gray', linestyle=':', linewidth=2, label='Batas Prediksi')
plt.text(last_date, plt.ylim()[0] * 1.05, '  Today', fontsize=10, color='gray')

plt.title(f'Prediksi Harga Bitcoin {FUTURE_DAYS} Hari ke Depan', fontsize=15, fontweight='bold')
plt.xlabel('Tanggal', fontsize=12)
plt.ylabel('Harga Bitcoin (USD)', fontsize=12)
plt.legend(fontsize=11)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
plt.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('08_prediksi_masa_depan.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualisasi prediksi masa depan selesai!")

In [ ]:
# ============================================================
# VISUALISASI METRIK KOMPARASI (BAR CHART)
# ============================================================

labels   = ['Train', 'Validation', 'Test']
rmse_vals = [metrics_train['RMSE'],     metrics_val['RMSE'],     metrics_test['RMSE']]
mae_vals  = [metrics_train['MAE'],      metrics_val['MAE'],      metrics_test['MAE']]
mape_vals = [metrics_train['MAPE'],     metrics_val['MAPE'],     metrics_test['MAPE']]
acc_vals  = [metrics_train['Accuracy'], metrics_val['Accuracy'], metrics_test['Accuracy']]

x    = np.arange(len(labels))
width = 0.3
colors = ['#3498DB', '#2ECC71', '#E74C3C']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Perbandingan Metrik Evaluasi Model GRU', fontsize=16, fontweight='bold')

metric_data = [
    (axes[0, 0], rmse_vals, 'RMSE (USD)',      'RMSE',      '${:,.0f}'),
    (axes[0, 1], mae_vals,  'MAE (USD)',        'MAE',       '${:,.0f}'),
    (axes[1, 0], mape_vals, 'MAPE (%)',         'MAPE',      '{:.4f}%'),
    (axes[1, 1], acc_vals,  'Akurasi (%)',      'Accuracy',  '{:.2f}%'),
]

for ax, values, title, ylabel, fmt in metric_data:
    bars = ax.bar(labels, values, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() * 1.01,
                fmt.format(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
    if 'Akurasi' in title:
        ax.set_ylim(0, 105)
        ax.axhline(y=80, color='orange', linestyle='--', alpha=0.7, label='Threshold 80%')
        ax.legend()

plt.tight_layout()
plt.savefig('09_metrik_komparasi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualisasi metrik selesai!")

In [ ]:
# ============================================================
# RINGKASAN AKHIR
# ============================================================

print()
print("RINGKASAN FINAL MODEL LSTM PREDIKSI BITCOIN")
print("=" * 65)
print(f"  Ticker             : BTC-USD (Yahoo Finance)")
print(f"  Periode Data       : {df['Date'].min().date()} s/d {df['Date'].max().date()}")
print(f"  Total Data         : {len(df):,} hari")
print(f"  Lookback Window    : {LOOKBACK} hari")
print(f"  Fitur Input        : {len(FEATURES)} fitur")
print()
print("  ARSITEKTUR LSTM:")
print(f"    Layer 1  : LSTM(128) + BatchNorm + Dropout(0.3)")
print(f"    Layer 2  : LSTM(64)  + BatchNorm + Dropout(0.3)")
print(f"    Layer 3  : LSTM(32)  + BatchNorm + Dropout(0.15)")
print(f"    Output   : Dense(16, relu) → Dense(1)")
print(f"    Optimizer: Adam (lr=0.001)")
print(f"    Loss     : MSE")
print()
print("  HASIL EVALUASI TEST SET:")
print(f"    RMSE     : ${metrics_test['RMSE']:>12,.2f}")
print(f"    MAE      : ${metrics_test['MAE']:>12,.2f}")
print(f"    MAPE     : {metrics_test['MAPE']:>11.4f} %")
print(f"    AKURASI  : {metrics_test['Accuracy']:>11.4f} %")
print()
print(f"  PREDIKSI BESOK (D+1): ${future_prices[0]:,.2f}")
print(f"  PREDIKSI 7 HARI     : ${future_prices[6]:,.2f}")
print(f"  PREDIKSI 30 HARI    : ${future_prices[-1]:,.2f}")

# Interpretasi akurasi
acc = metrics_test['Accuracy']
if acc >= 95:
    grade = "SANGAT BAIK"
elif acc >= 90:
    grade = "BAIK"
elif acc >= 80:
    grade = "CUKUP BAIK"
elif acc >= 70:
    grade = "PERLU IMPROVEMENT"
else:
    grade = "PERLU PERBAIKAN SIGNIFIKAN"

print(f"\n  GRADE AKURASI: {grade}")
print()
print("=" * 65)